During the previous validation phase, I identified various anomalies, inconsistencies, and business-specific outliers. In this section, I will evaluate each issue individually to determine whether it needs to be corrected, removed, maintained, or documented, considering its potential impact on business understanding and subsequent analysis.

I'm not touching the original (raw) DataFrames in the `data_understanding` section. Instead, I'll reread the data from the CSVs in the cleanup notebook and use the cleaned versions with new variable names. This is because the raw data should always remain unchanged and traceable. This way, I can see what I changed and why.
I'm uploading the raw data as well.

In [2]:
import pandas as pd

customers = pd.read_csv("../data/raw_data/olist/olist_customers_dataset.csv")
geo_locations = pd.read_csv("../data/raw_data/olist/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw_data/olist/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw_data/olist/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw_data/olist/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw_data/olist/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw_data/olist/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw_data/olist/olist_sellers_dataset.csv")
product_category_name_translations = pd.read_csv("../data/raw_data/olist/product_category_name_translation.csv")
closed_deals = pd.read_csv("../data/raw_data/marketing/olist_closed_deals_dataset.csv")
marketing_leads = pd.read_csv("../data/raw_data/marketing/olist_marketing_qualified_leads_dataset.csv")

BURADAN AŞAĞISI CLAUDE

I converted the date columns in the orders table to datetime.

In [3]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [4]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


I did the same thing for order_reviews as well.

In [5]:
new_date = ['review_creation_date', 'review_answer_timestamp']
for item in new_date:
    order_reviews[item] = pd.to_datetime(order_reviews[item])

In [6]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  str           
 1   order_id                 99224 non-null  str           
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  str           
 4   review_comment_message   40977 non-null  str           
 5   review_creation_date     99224 non-null  datetime64[us]
 6   review_answer_timestamp  99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 5.3 MB


In [7]:
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str)
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str)

In [8]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB


In [9]:
marketing_leads['first_contact_date'] = pd.to_datetime(marketing_leads['first_contact_date'])
closed_deals['won_date'] = pd.to_datetime(closed_deals['won_date'])

I used info() to check if data type conversions were successful.

There are rows that appear to be delivered but have an empty delivery date column. I'll remove these because there are only 8 records (i.e., 0.008% of the total rows), so it won't affect the analysis.

In [10]:
orders[(orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


That's why I cleaned this data. In the future, when I make a change to `orders_clean`, pandas might not know whether I'm changing it to the original "orders" or the new dataframe. So I'm adding `.copy()`. This creates a new, independent dataframe, unrelated to the original. It's a security measure so I can safely make changes to `orders_clean` later.

In [11]:
orders_clean = orders[~((orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isnull()))].copy()

Products priced under $1 (3 rows) were investigated and appear to be legitimate low-cost items, not data errors. Kept as-is.

There were payments with a "payment_type" of "undefined". To decide about these records, I will examine the payment amounts. If the payment amount is not 0, it means "payment exists but type is not specified", and I can give the payment_type a "unknown" tag.

In [12]:
order_payments[order_payments['payment_type'] == 'not_defined']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


However, the payment_value values ​​here are also 0. This means these lines don't represent a real monetary transaction. It could be a failed/canceled payment attempt, or the system may have created an empty record. Instead of deleting it, I'll mark the payment_type as "unknown" because since the payment is already 0, it won't affect any total/average calculations. Deleting it might cause discrepancies in other analyses (such as order count).

In [13]:
order_payments_clean = order_payments.copy()
order_payments_clean.loc[order_payments_clean['payment_type'] == 'not_defined', 'payment_type'] = 'unknown'

There are products where both weight and size are entered as 0. I will fill these in with the median. Because these products are real, only one field is missing/incorrectly entered.

There are products where both weight and size are entered as 0. I will fill these in with the median. Because these products are real, only one field is missing/incorrectly entered. 


In [14]:
median_weight = products[products['product_category_name'] == 'cama_mesa_banho']['product_weight_g'].median()
products.loc[products['product_weight_g'] == 0, 'product_weight_g'] = median_weight

I verified it again.

In [15]:
products.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.624237,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4281.980200,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,2.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


In [16]:
products.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.624237,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4281.980200,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,2.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


As I saw when examining the table, the products_category_name column has 610 empty values, which is 1.85% of the total rows. Filling it with the modulo operator isn't correct because it leads to errors in an analysis like "most frequently sold category". Therefore, I should mark it as "unknown/missing". This way, the data isn't lost, and it's indicated that the product category is unknown.

In [17]:
products_clean = products.copy()
products_clean['product_category_name'] = products_clean['product_category_name'].fillna('unknown')
products_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32951 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


I filtered out rows where `product_name_length` was empty, then checked how many times each value appeared in the `product_category_name` column in those rows.

All 610 rows were marked "unknown". This means that the missing elements in these three columns (product_name_length, product_description_length, product_photos_qty) occur in the same rows as the missing category elements.

It's likely that the seller didn't enter any details when registering these 610 products, meaning they are probably incomplete/incomplete product records.

In [18]:
products_clean[products_clean['product_name_lenght'].isnull()]['product_category_name'].value_counts(dropna=False)

product_category_name
unknown    610
Name: count, dtype: int64

Although the "product_name_length, product_description_length, product_photos_qty" columns are missing, I'm not deleting those rows; I can use the size/weight data. Instead, I'm filling them with the global median.

In [19]:
for col in ['product_name_lenght', 'product_description_lenght', 'product_photos_qty']:
    products_clean[col] = products_clean[col].fillna(products_clean[col].median())

In [20]:
order_reviews_clean = order_reviews.copy()
order_reviews_clean['review_creation_date'] = pd.to_datetime(order_reviews_clean['review_creation_date'])
order_reviews_clean['review_answer_timestamp'] = pd.to_datetime(order_reviews_clean['review_answer_timestamp'])

In [21]:
dup_order_ids = order_reviews_clean['order_id'].value_counts()
dup_order_ids = dup_order_ids[dup_order_ids > 1].index
order_reviews_clean[order_reviews_clean['order_id'].isin(dup_order_ids)].sort_values(['order_id', 'review_creation_date']).head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25,2017-08-29 21:45:57
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29,2017-08-30 01:59:12
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:47
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21,2018-03-22 01:32:08
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29,2018-03-30 03:16:19
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14,2017-07-17 13:58:06
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19,2017-07-20 12:06:11
